In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import random_split, Subset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torchvision.models import resnet18
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
import shutup

Загрузка, преобразование и разделение датасета на train, валидацию и test

In [2]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset_path = './archive/archive/simpsons_dataset'
train_dataset = ImageFolder(root=dataset_path, transform=train_transform)
test_dataset = ImageFolder(root=dataset_path, transform=test_transform)

print(f"Найдено классов: {len(train_dataset.classes)}")
print(f"Всего изображений: {len(train_dataset)}")

# Разделение на train/valid/test
train_size = int(0.6 * len(train_dataset))
valid_size = int(0.2 * len(train_dataset))
test_size = len(train_dataset) - train_size - valid_size

indices = list(range(len(train_dataset)))
train_indices, valid_indices, test_indices = random_split(indices, [train_size, valid_size, test_size])

train_subset = Subset(train_dataset, train_indices)
valid_subset = Subset(test_dataset, valid_indices)
test_subset = Subset(test_dataset, test_indices)

train_loader = DataLoader(train_subset, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
valid_loader = DataLoader(valid_subset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_subset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

Найдено классов: 42
Всего изображений: 20933


<h8>Модель для обучения<h8>

In [3]:
class SimpsonModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        backbone = resnet18(pretrained=True)
        self.backbone = nn.Sequential(*list(backbone.children())[:-1])
        self.bn = nn.BatchNorm1d(512)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(512, num_classes)
        
    def forward(self, x):
        x = self.backbone(x)
        x = x.view(x.size(0), -1)
        x = self.bn(x)
        x = self.dropout(x)
        return self.classifier(x)

<h8>Метрика<h8>

In [4]:
def calculate_f1(labels, predictions, num_classes, average='weighted'):
    labels_np = labels.cpu().numpy()
    predictions_np = predictions.cpu().numpy()
    
    f1 = f1_score(labels_np, predictions_np, average=average, zero_division=0)
    precision = precision_score(labels_np, predictions_np, average=average, zero_division=0)
    recall = recall_score(labels_np, predictions_np, average=average, zero_division=0)
    
    return f1, precision, recall

<h8>Обучение модели<h8>

In [5]:
def train_model(model, train_loader, valid_loader, optimizer, num_epochs, num_classes, output):
    best_valid_f1 = 0
    best_valid_precision = 0
    best_valid_recall = 0

    for epoch in range(num_epochs):
        print(f'\nЭпоха {epoch+1}/{num_epochs}')

        # Обучение
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0
        all_train_labels = []
        all_train_preds = []

        for images, labels in train_loader:
            images, labels = images.cuda(), labels.cuda()

            optimizer.zero_grad()
            logits = model(images)
            loss = F.cross_entropy(logits, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = logits.max(1)
            train_total += labels.size(0)
            train_correct += predicted.eq(labels).sum().item()

            all_train_labels.extend(labels.cpu())
            all_train_preds.extend(predicted.cpu())

        train_loss /= len(train_loader)
        train_acc = 100. * train_correct / train_total
        train_f1, _, _ = calculate_f1(torch.tensor(all_train_labels), torch.tensor(all_train_preds), num_classes)

        # Валидация
        model.eval()
        valid_loss = 0
        valid_correct = 0
        valid_total = 0
        all_valid_labels = []
        all_valid_preds = []

        with torch.no_grad():
            for images, labels in valid_loader:
                images, labels = images.cuda(), labels.cuda()
                logits = model(images)
                loss = F.cross_entropy(logits, labels)

                valid_loss += loss.item()
                _, predicted = logits.max(1)
                valid_total += labels.size(0)
                valid_correct += predicted.eq(labels).sum().item()

                all_valid_labels.extend(labels.cpu())
                all_valid_preds.extend(predicted.cpu())

        valid_loss /= len(valid_loader)
        valid_acc = 100. * valid_correct / valid_total

        valid_f1, valid_precision, valid_recall = calculate_f1(torch.tensor(all_valid_labels), torch.tensor(all_valid_preds), num_classes, 
                                                               average='weighted')

        output.add_scalars('Loss', {'train': train_loss, 'valid': valid_loss}, epoch)
        output.add_scalars('F1', {'train': train_f1, 'valid': valid_f1}, epoch)

        print(f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%, Train F1: {train_f1:.4f}')
        print(f'Valid Loss: {valid_loss:.4f}, Valid Acc: {valid_acc:.2f}%')
        print(f'Valid F1: {valid_f1:.4f}, Precision: {valid_precision:.4f}, Recall: {valid_recall:.4f}')

        if valid_f1 > best_valid_f1:
            best_valid_f1 = valid_f1
            torch.save(model.state_dict(), 'best_model.pth')

        if valid_precision > best_valid_precision:
            best_valid_precision = valid_precision
            torch.save(model.state_dict(), 'best_model_precision.pth')

        if valid_recall > best_valid_recall:
            best_valid_recall = valid_recall
            torch.save(model.state_dict(), 'best_model_recall.pth')

<h8>Тест<h8>

In [53]:
def test_model(model, test_loader, num_classes):
    print("Test")

    model.load_state_dict(torch.load('best_model.pth'))
    model.eval()

    test_loss = 0
    test_correct = 0
    test_total = 0
    all_test_labels = []
    all_test_preds = []

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.cuda(), labels.cuda()
            logits = model(images)
            loss = F.cross_entropy(logits, labels)

            test_loss += loss.item()
            _, predicted = logits.max(1)
            test_total += labels.size(0)
            test_correct += predicted.eq(labels).sum().item()

            all_test_labels.extend(labels.cpu())
            all_test_preds.extend(predicted.cpu())

    test_loss /= len(test_loader)
    test_acc = 100. * test_correct / test_total

    test_f1_weighted, test_precision, test_recall = calculate_f1(torch.tensor(all_test_labels), torch.tensor(all_test_preds), num_classes,
                                                                 average='weighted')
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Accuracy: {test_acc:.2f}%")
    print(f"Test F1 (weighted): {test_f1_weighted:.4f}")
    print(f"Precision: {test_precision:.4f}")
    print(f"Recall: {test_recall:.4f}")

    print(classification_report(all_test_labels, all_test_preds, zero_division=0))

In [7]:
shutup.please()

output = SummaryWriter('log')
class_names = train_dataset.classes
num_classes = len(train_dataset.classes)
model = SimpsonModel(num_classes=num_classes).cuda()

optimizer = torch.optim.Adam(model.parameters(), lr=5e-5, weight_decay=1e-3)

train_model(
    model=model,
    train_loader=train_loader,
    valid_loader=valid_loader,
    optimizer=optimizer,
    num_epochs=6,
    num_classes=num_classes,
    output=output
)
output.close()


Эпоха 1/6
Train Loss: 1.6733, Train Acc: 63.36%, Train F1: 0.6410
Valid Loss: 0.6178, Valid Acc: 87.46%
Valid F1: 0.8502, Precision: 0.8501, Recall: 0.8746

Эпоха 2/6
Train Loss: 0.4772, Train Acc: 89.97%, Train F1: 0.8848
Valid Loss: 0.3697, Valid Acc: 91.81%
Valid F1: 0.9073, Precision: 0.9092, Recall: 0.9181

Эпоха 3/6
Train Loss: 0.2841, Train Acc: 93.96%, Train F1: 0.9327
Valid Loss: 0.2607, Valid Acc: 94.34%
Valid F1: 0.9379, Precision: 0.9410, Recall: 0.9434

Эпоха 4/6
Train Loss: 0.1865, Train Acc: 96.33%, Train F1: 0.9601
Valid Loss: 0.2208, Valid Acc: 95.25%
Valid F1: 0.9507, Precision: 0.9525, Recall: 0.9525

Эпоха 5/6
Train Loss: 0.1286, Train Acc: 97.64%, Train F1: 0.9749
Valid Loss: 0.1912, Valid Acc: 95.75%
Valid F1: 0.9565, Precision: 0.9574, Recall: 0.9575

Эпоха 6/6
Train Loss: 0.0901, Train Acc: 98.46%, Train F1: 0.9842
Valid Loss: 0.1709, Valid Acc: 96.15%
Valid F1: 0.9608, Precision: 0.9616, Recall: 0.9615


In [11]:
%load_ext tensorboard
%tensorboard --logdir log

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [54]:
shutup.please()
test_model(model=model, test_loader=test_loader, num_classes=num_classes)

Test
Test Loss: 0.1528
Test Accuracy: 96.49%
Test F1 (weighted): 0.9642
Precision: 0.9650
Recall: 0.9649
              precision    recall  f1-score   support

           0       0.98      0.98      0.98       176
           1       1.00      0.89      0.94         9
           2       0.94      1.00      0.97       122
           3       0.88      0.79      0.83        19
           4       0.96      0.99      0.97       279
           5       0.96      0.96      0.96        25
           6       0.94      0.96      0.95       246
           7       0.97      0.96      0.96       212
           8       0.88      0.78      0.82         9
           9       0.95      0.95      0.95       102
          10       0.00      0.00      0.00         1
          11       0.97      0.97      0.97        88
          12       1.00      0.29      0.44         7
          13       1.00      1.00      1.00         2
          14       0.90      0.93      0.92        30
          15       0.98      0